# Notebook to test priors added via prior-repos

## TabPFN v1 prior

The repo is added as a submodule. The documentation can be found at
[https://github.com/automl/tabpfn-v1-prior](https://github.com/automl/tabpfn-v1-prior).

In [ ]:
import torch
from tabpfn_prior import build_tabpfn_prior

prior = build_tabpfn_prior(
    prior_type='mlp',
    num_steps=1000, # how many steps until prior is exhausted
    batch_size=1, # batch size, all datasets in one batch have the same num features, num datapoints and num classes
    num_datapoints_max=1000,
    num_features=11,
    max_num_classes=5,
    device='cpu',
    flexible=True,
    return_categorical_mask=True,
    differentiable=True,
    nan_handling=True,
)

# Generate synthetic data
for batch in prior:
    x = batch['x']          # Input features [batch_size, seq_len, num_features]
    y = batch['y']          # Labels [batch_size, seq_len]
    target_y = batch['target_y']  # Target labels (same as y)
    eval_pos = batch['single_eval_pos']  # Evaluation position
    categorical_mask = batch.get('categorical_mask', None)  # Categorical feature mask
    
    for i in range(x.shape[0]):
        print(f"Sample {i}: x.shape={x[i].shape}, y.shape={y[i].shape}, unique_classes={torch.unique(y[i])}, first 5 x: {x[i][:5]}, first 5 y: {y[i][:5]}")
        print(f"Number of nans in x: {torch.isnan(x[i]).sum().item()}")
        if torch.isnan(x[i]).sum().item() > 0:
            print(f"Indices of nans in x: {torch.nonzero(torch.isnan(x[i]), as_tuple=True)}")
        if categorical_mask is not None and categorical_mask[i].sum() > 0:
            print(f"Categorical features at indices: {torch.nonzero(categorical_mask[i], as_tuple=True)[0].tolist()}")
    break


print(f"Generated data: x.shape={x.shape}, y.shape={y.shape}")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

cat_counts = []

for batch in prior:
    categorical_mask = batch.get('categorical_mask', None)
    if categorical_mask is not None:
        for i in range(categorical_mask.shape[0]):
            num_cat = int(categorical_mask[i].sum().item())
            cat_counts.append(num_cat)

# Count occurrences
count_dist = Counter(cat_counts)
print("Distribution of number of categorical features:", count_dist)

# Plot distribution
plt.bar(count_dist.keys(), count_dist.values())
plt.xlabel('Number of categorical features')
plt.ylabel('Frequency')
plt.title('Distribution over categorical features per dataset')
plt.show()

In [ ]:
missing_fractions = []

for batch in prior:
    x = batch['x']
    for i in range(x.shape[0]):
        total = x[i].numel()
        missing = torch.isnan(x[i]).sum().item()
        missing_frac = missing / total
        missing_fractions.append(missing_frac)

rounded_fractions = [round(f, 2) for f in missing_fractions]
count_dist = Counter(rounded_fractions)
print("Distribution of missing value fractions (rounded):", count_dist)

plt.hist(missing_fractions, bins=20)
plt.xlabel('Fraction of missing values')
plt.ylabel('Frequency')
plt.title('Distribution of missing value fraction per dataset')
plt.show()

## tabularpriors

This repo is added as a submodule. The documentation can be found at
[https://github.com/automl/tabularpriors](https://github.com/automl/tabularpriors).

In [ ]:
from tabularpriors.dataloader import TabICLPriorDataLoader

prior = TabICLPriorDataLoader(
    num_steps=20100,
    batch_size=8,
    num_datapoints_min=1,
    num_datapoints_max=1024,
    min_features=3,
    max_features=10,
    max_num_classes=5,
    device='cpu'
)

# Generate synthetic data
for batch in prior:
    x = batch['x']          # Input features [batch_size, seq_len, num_features]
    y = batch['y']          # Labels [batch_size, seq_len]
    target_y = batch['target_y']  # Target labels (same as y)
    eval_pos = batch['single_eval_pos']  # Evaluation position
    break

print(f"Generated data: x.shape={x.shape}, y.shape={y.shape}")

In [ ]:
from tabularpriors.dataloader import TICLPriorDataLoader
from ticl.priors import BagPrior
from ticl.model_configs import get_prior_config
from ticl.priors import ClassificationAdapterPrior, BagPrior, BooleanConjunctionPrior, StepFunctionPrior
import ticl.priors as priors

max_features = 10
n_samples = 1000

prior_config = get_prior_config(max_features=max_features, n_samples=n_samples)['prior']

prior_type = prior_config['prior_type']
gp_flexible = ClassificationAdapterPrior(priors.GPPrior(prior_config['gp']), num_features=prior_config['num_features'], **prior_config['classification'])
mlp_flexible = ClassificationAdapterPrior(priors.MLPPrior(prior_config['mlp']), num_features=prior_config['num_features'], **prior_config['classification'])

if prior_type == 'prior_bag':
    # Prior bag combines priors
    prior = BagPrior(base_priors={'gp': gp_flexible, 'mlp': mlp_flexible},
                        prior_weights={'mlp': 0.961, 'gp': 0.039})
elif prior_type == "step_function":
    prior = priors.StepFunctionPrior(prior_config['step_function'])
elif prior_type == "boolean_only":
    prior = BooleanConjunctionPrior(hyperparameters=prior_config['boolean'])
elif prior_type == "bag_boolean":
    boolean = BooleanConjunctionPrior(hyperparameters=prior_config['boolean'])
    prior = BagPrior(base_priors={'gp': gp_flexible, 'mlp': mlp_flexible, 'boolean': boolean},
                        prior_weights={'mlp': 0.9, 'gp': 0.02, 'boolean': 0.08})
else:
    raise ValueError(f"Prior type {prior_type} not supported.")
    

prior = TICLPriorDataLoader(
    prior=prior,
    num_steps=20100,
    batch_size=8,
    num_datapoints_max=n_samples, # n_samples
    num_features=10,
    device='cuda',
)

# Generate synthetic data
for batch in prior:
    x = batch['x']          # Input features [batch_size, seq_len, num_features]
    y = batch['y']          # Labels [batch_size, seq_len]
    target_y = batch['target_y']  # Target labels (same as y)
    eval_pos = batch['single_eval_pos']  # Evaluation position
    break

print(f"Generated data: x.shape={x.shape}, y.shape={y.shape}")